# Thunder Compute Notebook — SigLLM (MF + Q-Former + Frozen LLM for Yes/No)

Migrated from `SigLLM._best_24_05_2026ipynb.ipynb` (Colab version). Key differences:
- No Google Drive — uses Thunder's persistent home directory (`~`).
- No fresh miniconda install at `/usr/local` — expects conda in PATH or installs once to `~/miniconda3`.
- Paths parameterized via `WORKDIR` / `REPO_DIR` so you can switch between `~`, `/workspace`, or `/persist` without find-replace.
- Backup section uses Hugging Face Hub or `tar` + Thunder snapshot instead of Drive copy.

**Before first run, verify**:
1. `WORKDIR` below matches your Thunder instance persistent path. Common defaults: `~` (home), `/workspace`, `/persist`. Check `df -h` to see which mount has the most space and persists.
2. `/ephemeral` is wiped on instance stop — do NOT store code or checkpoints there. Only use for scratch (e.g. HF cache).
3. GPU is visible: cell `[Verify GPU]` runs `nvidia-smi`.

# **0. Setup Environment**

## Configure paths (edit once, used everywhere)

In [ ]:
import os

# Adjust to match your Thunder Compute persistent mount.
# Common: '~' (home, default), '/workspace', '/persist'.
WORKDIR = os.path.expanduser('~')
REPO_DIR = f'{WORKDIR}/SigLLM'

# Conda env name (kept consistent with env.yaml)
ENV_NAME = 'sigllm'

# Export as shell env vars so '!' cells can use $WORKDIR / $REPO_DIR / $SIGLLM_ROOT.
# SIGLLM_ROOT is read by configs/config.yaml via OmegaConf ${oc.env:SIGLLM_ROOT,...}
# interpolation, so every Stage path resolves to this repo without /content symlinks.
os.environ['WORKDIR'] = WORKDIR
os.environ['REPO_DIR'] = REPO_DIR
os.environ['SIGLLM_ROOT'] = REPO_DIR
os.environ['ENV_NAME'] = ENV_NAME

# HF + tokenizer caches — put on persistent disk to survive restarts (skip Thunder /ephemeral)
os.environ['HF_HOME'] = f'{WORKDIR}/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{WORKDIR}/.cache/huggingface/transformers'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'WORKDIR      = {WORKDIR}')
print(f'REPO_DIR     = {REPO_DIR}')
print(f'SIGLLM_ROOT  = {os.environ["SIGLLM_ROOT"]}')
print(f'ENV_NAME     = {ENV_NAME}')
print(f'HF_HOME      = {os.environ["HF_HOME"]}')

## Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
# Sanity: free disk on WORKDIR (need >= ~80GB for full pipeline + Qwen2-7B + checkpoints)
!df -h $WORKDIR

In [ ]:
import os
from getpass import getpass

# Skip clone if repo already exists (persistent disk — only clone first session)
if not os.path.isdir(REPO_DIR):
    token = getpass('Enter your GitHub Personal Access Token (PAT): ')
    repo_url = f'https://{token}@github.com/QuocBaoBuiNguyen/SigLLM.git'
    os.chdir(WORKDIR)
    !git clone {repo_url}
else:
    print(f'Repo already present at {REPO_DIR} — skipping clone')

os.chdir(REPO_DIR)
!git checkout develop
!ls -a

In [ ]:
%cd $REPO_DIR
!git checkout develop && git pull origin develop

In [ ]:
!git branch
# Adjust to active feature branch when needed
!git checkout feat/swap-llm-qwen2 && git pull origin feat/swap-llm-qwen2

## Python path + autoreload

In [ ]:
import sys
import importlib
from types import ModuleType

# Shim for legacy `import imp` (deprecated in py3.12)
imp = ModuleType('imp')
imp.reload = importlib.reload
sys.modules['imp'] = imp

%reload_ext autoreload

In [ ]:
%env PYTHONPATH=src:$PYTHONPATH

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Setup conda environment

Thunder Compute images often ship without conda. The cell below installs miniconda to `~/miniconda3` only if `conda` is not already on PATH — idempotent across restarts.

In [ ]:
%%bash
set -e

if command -v conda > /dev/null 2>&1; then
    echo "conda already on PATH: $(command -v conda)"
    conda --version
    exit 0
fi

MINICONDA_PREFIX="$HOME/miniconda3"
if [ -x "$MINICONDA_PREFIX/bin/conda" ]; then
    echo "Found existing miniconda at $MINICONDA_PREFIX — adding to PATH for this session"
else
    echo "Installing miniconda to $MINICONDA_PREFIX..."
    INSTALLER="/tmp/Miniconda3.sh"
    wget -q -c https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O "$INSTALLER"
    bash "$INSTALLER" -b -p "$MINICONDA_PREFIX"
fi

# Best-effort: persist conda init in ~/.bashrc so future shell sessions auto-load it.
# Skip silently if .bashrc is read-only / owned by another user — not fatal for Jupyter.
if [ -w "$HOME/.bashrc" ] || [ ! -e "$HOME/.bashrc" ]; then
    if ! grep -q 'miniconda3/etc/profile.d/conda.sh' "$HOME/.bashrc" 2>/dev/null; then
        echo 'source $HOME/miniconda3/etc/profile.d/conda.sh' >> "$HOME/.bashrc" \n            && echo "Added conda init to ~/.bashrc" \n            || echo "WARN: could not append to ~/.bashrc — skipping (not fatal for Jupyter)."
    fi
else
    echo "WARN: ~/.bashrc not writable by $(whoami) — skipping bashrc init (not fatal for Jupyter)."
fi

source "$MINICONDA_PREFIX/etc/profile.d/conda.sh"
conda --version

In [ ]:
# Accept Anaconda channel ToS (silent failure if already accepted)
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true

In [ ]:
%%bash
set -e
source $HOME/miniconda3/etc/profile.d/conda.sh

ENV_PREFIX="$HOME/miniconda3/envs/${ENV_NAME}"
ENV_YAML="${REPO_DIR}/env.yaml"

if [ -d "$ENV_PREFIX" ] && [ -f "$ENV_PREFIX/bin/python" ]; then
    echo "Env '${ENV_NAME}' already exists at $ENV_PREFIX — skipping create."
else
    echo "Creating env '${ENV_NAME}' from $ENV_YAML ..."
    conda env create -f "$ENV_YAML"
fi

echo "Done."

In [ ]:
# Pin transformers + accelerate to versions tested with this branch
!source $HOME/miniconda3/etc/profile.d/conda.sh && \
conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python -m pip install -U \
"transformers==4.38.2" \
"accelerate==0.27.2"

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda env list

### Run this only if you change `env.yaml`

In [ ]:
# !source $HOME/miniconda3/etc/profile.d/conda.sh && conda env update --file ${REPO_DIR}/env.yaml --prune

# **1. Dataset Preparation**

## Download and prepare datasets

In [ ]:
import sys
sys.path.append(f'{REPO_DIR}/src')

In [ ]:
!rm -rf $REPO_DIR/data/processed/

In [ ]:
!unzip -o $REPO_DIR/data/raw/ml-1m.zip -d $REPO_DIR/data/raw/

## Primary ML-1M preprocessing
Handles temporal splitting, ID remapping, and sequential history construction.

> **Note**: `src/sigllm/datasets/data_preprocessing.py` hard-codes Colab paths (`/content/...`). If you keep `WORKDIR = ~`, edit that script to use `os.path.expanduser('~/SigLLM/...')` or pass paths via env var. Quick check after run: `ls $REPO_DIR/data/processed/ml-1m/` should show `train_ood2.pkl`, `valid_ood2.pkl`, `test_ood2.pkl`.

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/datasets/data_preprocessing.py

### **Note**: cells below are for visualizing / exploring the prepared data only.

In [ ]:
from sigllm.datasets import build_ml1m
import pandas as pd

data_dir = f'{REPO_DIR}/data/processed/ml-1m'
train_df = pd.read_pickle(f'{data_dir}/train_ood2.pkl')
valid_df = pd.read_pickle(f'{data_dir}/valid_ood2.pkl')
test_df = pd.read_pickle(f'{data_dir}/test_ood2.pkl')
print(f'train: {train_df.shape}, valid: {valid_df.shape}, test: {test_df.shape}')
train_df.head()

## Evaluation-specific tagging
Segments the test set into "Warm" (high frequency) and "Strict-Cold" (unseen) groups.

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/datasets/preprocess_test_cold_warm.py

In [ ]:
from sigllm.datasets import process_warm_cold
import pandas as pd

wc_df = pd.read_pickle(f'{REPO_DIR}/data/processed/ml-1m/test_warm_cold_ood2.pkl')
print(wc_df[['warm', 'cold']].sum())
wc_df.head()

## Q-Former Dataset

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/pipelines/multimodal/build_qformer_dataset.py \
--cfg-path $REPO_DIR/configs/config.yaml

# **2. Training Stages (3 steps)**

In [ ]:
!mkdir -p $REPO_DIR/ckpt/llm/base/
!mkdir -p $REPO_DIR/ckpt/mf/
!mkdir -p $REPO_DIR/ckpt/qformer_stage3_step1_lora/qwen2-7b-lora/

## Pull base LLM weights (Qwen2-7B-Base)

Downloads to `$REPO_DIR/ckpt/llm/qwen2-7b-base/`. ~15GB, persists across sessions if you keep WORKDIR on persistent disk.

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/pipelines/llm/pull_llm_model.py

## Stage 0 — Train MF (Collaborative Filtering backbone)

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/pipelines/rec/train_rec_baseline.py

## Stage 1 — Representation Pretraining (Q-Former + projector, no Yes/No forcing yet)

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/pipelines/multimodal/train_qformer_stage1_representation.py \
--cfg-path $REPO_DIR/configs/config.yaml

## Stage 2 — Generative pretraining

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
python $REPO_DIR/src/sigllm/pipelines/multimodal/train_qformer_stage2_generative.py \
--cfg-path $REPO_DIR/configs/config.yaml

## Stage 3 — Yes/No Fine-tuning with Frozen LLM + LoRA

Step 1: train LoRA on text-only prompt. Step 2: train Q-Former + projection on full prompt with LoRA frozen.

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && \
conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
export TOKENIZERS_PARALLELISM=false && \
python $REPO_DIR/src/sigllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
--cfg-path $REPO_DIR/configs/config.yaml

In [ ]:
!source $HOME/miniconda3/etc/profile.d/conda.sh && \
conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
export TOKENIZERS_PARALLELISM=false && \
python $REPO_DIR/src/sigllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
--cfg-path $REPO_DIR/configs/config.yaml

# **3. Evaluation**

## 3.1 Full evaluation (normal — numbers for thesis report)

In [ ]:
%cd $REPO_DIR
!source $HOME/miniconda3/etc/profile.d/conda.sh && \
conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
export TOKENIZERS_PARALLELISM=false && \
python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie \
--cfg-path configs/config.yaml \
--options run.evaluate=True

## 3.2 Ablation (`ablate_soft_tokens=True`)

Measures Q-Former marginal contribution by zeroing soft tokens. Pair with 3.1 for the ablation table.

In [ ]:
%cd $REPO_DIR
!source $HOME/miniconda3/etc/profile.d/conda.sh && \
conda activate $ENV_NAME && export SIGLLM_ROOT=$REPO_DIR && \
export TOKENIZERS_PARALLELISM=false && \
python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie \
--cfg-path configs/config.yaml \
--options model.ablate_soft_tokens=True run.evaluate=True

# **4. Snapshot & terminate instance**

Thunder Compute's snapshot saves the full disk state to cloud storage. Workflow:
1. Create snapshot of the running instance (preserves ckpts, env, code, dataset).
2. Verify snapshot succeeded.
3. Delete the instance to stop GPU billing — snapshot survives.
4. Later, restore from snapshot to resume work.

Run cells below from the **instance itself** (`tnr` CLI is pre-installed on Thunder VMs) OR from your **local machine** if you have `tnr` configured locally. Local is safer — if you run from inside the instance, the `tnr delete` step will kill its own session mid-way.

## 4.1 List instances + verify snapshot CLI syntax

Run `tnr snapshot --help` once to confirm syntax on your Thunder version. Docs are sparse; the script below uses the most common form `tnr snapshot <instance-id>` — adjust if your CLI uses `tnr snapshot create <id>` instead.

In [ ]:
!tnr status
!tnr snapshot --help 2>&1 | head -30

## 4.2 Snapshot-and-delete script

The script `scripts/thunder_snapshot_and_delete.sh` is the safe way to do this from **your local machine**. It:
1. Takes an instance ID as argument.
2. Triggers snapshot.
3. Polls until snapshot completes.
4. Asks for confirmation (`yes` typed manually) before delete — unless `--yes` flag passed.
5. Deletes the instance.

Usage from local machine:
```bash
# Interactive (asks for confirm)
./scripts/thunder_snapshot_and_delete.sh 0

# Non-interactive (use carefully)
./scripts/thunder_snapshot_and_delete.sh 0 --yes
```

If you ran the script and want to re-verify it exists in the repo:

In [ ]:
!ls -l $REPO_DIR/scripts/thunder_snapshot_and_delete.sh 2>/dev/null || echo 'Script not present — git pull to fetch latest.'

## 4.3 Quick snapshot-only (from inside the instance, no delete)

Use this if you just want to checkpoint progress without terminating. Cheaper than keeping the GPU running for hours of idle time.

In [ ]:
# Edit the instance ID after running `tnr status`
INSTANCE_ID = 0  # change to match your instance
!tnr snapshot {INSTANCE_ID}